# Create the CSV and periodogram search

## Imports y funciones auxiliares

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits
import math
import os
import seaborn as sns
import re
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
from pandarallel import pandarallel
import lightkurve as lk
import pandas as pd
from scipy.signal import find_peaks
from astropy.io import fits
#import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import re
import os 
from astropy.timeseries import LombScargle
import math

pandarallel.initialize(nb_workers=30, progress_bar=True)

INFO: Pandarallel will run on 30 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [2]:
def open_LC(TIC, sector, path_download='/home/nicolas/nico/git/OB-stars_LC/dowload_parallell/'):
    TIC = str(TIC).zfill(16)
    sector = str(sector).zfill(4)
    path_lc = f'hlsp_tess-spoc_tess_phot_{TIC}-s{sector}_tess_v1_lc.fits'   
    lcf = lk.read(path_download+path_lc, quality_bitmask="hardest", flux_column="pdcsap_flux")    
    lc = lcf.remove_nans()
    time = np.array(lc.time.value)
    flux = np.array(lc.flux.value)
    flux_err = np.array(lc.flux_err.value)
    
    flux_new = convert_to_little_endian(flux)
    time_new = convert_to_little_endian(time)
    flux_err_new = convert_to_little_endian(flux_err)

    df = pd.DataFrame({"Time":np.array(time_new),"flux":np.array(flux_new),"flux_err":np.array(flux_err_new)})
    return df




    def make_2d_histogram(n_bins_x, n_bins_y, data_mag, data_fase, norm_max="max"):
        bins_x = np.linspace(0, 1, n_bins_x)
        bins_y = np.linspace(data_mag.min(), data_mag.max(), n_bins_y)
    
        hist_data, _xbins, _ybins = np.histogram2d(
            data_fase, data_mag, bins=(bins_x, bins_y)
        )
    
        if norm_max == "max":
            max_value = hist_data.max()
            hist_data_norm = hist_data / max_value 
    
        elif norm_max == "log":
            positive = hist_data > 0
    
            if not np.any(positive):
                raise ValueError("El histograma no contiene valores positivos para aplicar log.")
    
            vmin = hist_data[positive].min()
            vmax = hist_data.max()
    
            if vmin == vmax:
                raise ValueError("vmin == vmax; no se puede normalizar en escala log.")
    
            hist_data_log = np.zeros_like(hist_data, dtype=float)
            hist_data_log[positive] = (
                np.log(hist_data[positive]) - np.log(vmin)
            ) / (np.log(vmax) - np.log(vmin))
    
            hist_data_norm = hist_data_log
    
        elif norm_max == "none":
            hist_data_norm = hist_data
    
        else:
            raise ValueError(
                f"norm_max='{norm_max}' no soportado. Usa 'max', 'log' o 'none'."
            )
    
        hist_data_transposed = hist_data_norm.T[::-1]
        hdu = fits.PrimaryHDU(data=hist_data_transposed)
        return hdu

    

import sys
from astropy.io import fits
def convert_to_little_endian(arr):
    # Si el array ya es little-endian, lo retornamos sin cambios
    if arr.dtype.byteorder == '<' or (arr.dtype.byteorder == '=' and sys.byteorder == 'little'):
        return arr
    # De lo contrario, cambiamos el orden de los bytes
    else:
        return arr.byteswap().newbyteorder('<')

In [3]:
def plot_classification_results(df_peaks, lc, model_softmax,tic,sector,clas):
    n = len(df_peaks)
    cols = math.ceil(np.sqrt(n))
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
    axes = axes.flatten()

    class_names = ['ELL', 'M', 'CEP', 'DST', 'E', 'LPV', 'RR', 'Rndm']
    unique_harmonics = sorted(df_peaks["harmonics"].unique())
    color_palette = sns.color_palette("colorblind", len(unique_harmonics))
    harmonic_color_map = {h: color_palette[i] for i, h in enumerate(unique_harmonics)}
    for i, ax in enumerate(axes):
        if i < n:
            per = df_peaks.per.values[i]
            power = df_peaks.power.values[i]
            harmonic = df_peaks.harmonics.values[i]
            fase = np.mod(lc.time.value, per) / per
            hdu = make_2d_histogram(33, 33, lc.flux.value, fase, norm_max="max")
            predict_results = model_softmax.predict(np.expand_dims(hdu.data, axis=(0, -1)))
            predict_results = pd.DataFrame(predict_results, columns=class_names)
            classif = predict_results.T.sort_values(by=0, ascending=False).head(1)[0].index[0]
            classif_prob = predict_results.T.sort_values(by=0, ascending=False).head(1)[0].values[0]

            ax.imshow(hdu.data, aspect='auto')
            ax.set_title(f'Period = {per:.4f} \n Power:{power:.4f} \n harmonic: {harmonic} \n class = {classif} \n Prob= {classif_prob:.4f}')
            # Colorear los bordes
            color = harmonic_color_map[harmonic]
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(6)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xticklabels([])
            ax.set_yticklabels([])
        else:
            ax.set_visible(False)

    plt.tight_layout()
    plt.savefig(f"example_{clas}_{tic}_{sector}.pdf",dpi=300)

In [4]:
from numpy import (
    sum as npsum, arange as nparange, correlate as npcorrelate,
    max as npmax, median as npmedian, array as nparray, abs as npabs
)
from astrobase.lcmath import fill_magseries_gaps

#####################
## AUTOCORRELATION ##
#####################
def _autocorr_func1(mags, lag, maglen, magmed, magstd):
    '''Calculates the autocorr of mag series for specific lag.

    This version of the function is taken from: Kim et al. (`2011
    <https://dx.doi.org/10.1088/0004-637X/735/2/68>`_)

    Parameters
    ----------

    mags : np.array
        This is the magnitudes array. MUST NOT have any nans.

    lag : float
        The specific lag value to calculate the auto-correlation for. This MUST
        be less than total number of observations in `mags`.

    maglen : int
        The number of elements in the `mags` array.

    magmed : float
        The median of the `mags` array.

    magstd : float
        The standard deviation of the `mags` array.

    Returns
    -------

    float
        The auto-correlation at this specific `lag` value.

    '''

    lagindex = nparange(1,maglen-lag)
    products = (mags[lagindex] - magmed) * (mags[lagindex+lag] - magmed)
    acorr = (1.0/((maglen - lag)*magstd)) * npsum(products)

    return acorr



def _autocorr_func2(mags, lag, maglen, magmed, magstd):
    '''
    This is an alternative function to calculate the autocorrelation.

    This version is from (first definition):

    https://en.wikipedia.org/wiki/Correlogram#Estimation_of_autocorrelations

    Parameters
    ----------

    mags : np.array
        This is the magnitudes array. MUST NOT have any nans.

    lag : float
        The specific lag value to calculate the auto-correlation for. This MUST
        be less than total number of observations in `mags`.

    maglen : int
        The number of elements in the `mags` array.

    magmed : float
        The median of the `mags` array.

    magstd : float
        The standard deviation of the `mags` array.

    Returns
    -------

    float
        The auto-correlation at this specific `lag` value.

    '''

    lagindex = nparange(0,maglen-lag)
    products = (mags[lagindex] - magmed) * (mags[lagindex+lag] - magmed)

    autocovarfunc = npsum(products)/lagindex.size
    varfunc = npsum(
        (mags[lagindex] - magmed)*(mags[lagindex] - magmed)
    )/mags.size

    acorr = autocovarfunc/varfunc

    return acorr



def _autocorr_func3(mags, lag, maglen, magmed, magstd):
    '''
    This is yet another alternative to calculate the autocorrelation.

    Taken from: `Bayesian Methods for Hackers by Cameron Pilon <http://nbviewer.jupyter.org/github/CamDavidsonPilon/Probabilistic-Programming-and-Bayesian-Methods-for-Hackers/blob/master/Chapter3_MCMC/Chapter3.ipynb#Autocorrelation>`_

    (This should be the fastest method to calculate ACFs.)

    Parameters
    ----------

    mags : np.array
        This is the magnitudes array. MUST NOT have any nans.

    lag : float
        The specific lag value to calculate the auto-correlation for. This MUST
        be less than total number of observations in `mags`.

    maglen : int
        The number of elements in the `mags` array.

    magmed : float
        The median of the `mags` array.

    magstd : float
        The standard deviation of the `mags` array.

    Returns
    -------

    float
        The auto-correlation at this specific `lag` value.

    '''

    # from http://tinyurl.com/afz57c4
    result = npcorrelate(mags, mags, mode='full')
    result = result / npmax(result)

    return result[int(result.size / 2):]



def autocorr_magseries(times, mags, errs,
                       maxlags=1000,
                       func=_autocorr_func3,
                       fillgaps=0.0,
                       filterwindow=11,
                       forcetimebin=None,
                       sigclip=3.0,
                       magsarefluxes=False,
                       verbose=True):
    '''This calculates the ACF of a light curve.

    This will pre-process the light curve to fill in all the gaps and normalize
    everything to zero. If `fillgaps = 'noiselevel'`, fills the gaps with the
    noise level obtained via the procedure above. If `fillgaps = 'nan'`, fills
    the gaps with `np.nan`.

    Parameters
    ----------

    times,mags,errs : np.array
        The measurement time-series and associated errors.

    maxlags : int
        The maximum number of lags to calculate.

    func : Python function
        This is a function to calculate the lags.

    fillgaps : 'noiselevel' or float
        This sets what to use to fill in gaps in the time series. If this is
        'noiselevel', will smooth the light curve using a point window size of
        `filterwindow` (this should be an odd integer), subtract the smoothed LC
        from the actual LC and estimate the RMS. This RMS will be used to fill
        in the gaps. Other useful values here are 0.0, and npnan.

    filterwindow : int
        The light curve's smoothing filter window size to use if
        `fillgaps='noiselevel`'.

    forcetimebin : None or float
        This is used to force a particular cadence in the light curve other than
        the automatically determined cadence. This effectively rebins the light
        curve to this cadence. This should be in the same time units as `times`.

    sigclip : float or int or sequence of two floats/ints or None
        If a single float or int, a symmetric sigma-clip will be performed using
        the number provided as the sigma-multiplier to cut out from the input
        time-series.

        If a list of two ints/floats is provided, the function will perform an
        'asymmetric' sigma-clip. The first element in this list is the sigma
        value to use for fainter flux/mag values; the second element in this
        list is the sigma value to use for brighter flux/mag values. For
        example, `sigclip=[10., 3.]`, will sigclip out greater than 10-sigma
        dimmings and greater than 3-sigma brightenings. Here the meaning of
        "dimming" and "brightening" is set by *physics* (not the magnitude
        system), which is why the `magsarefluxes` kwarg must be correctly set.

        If `sigclip` is None, no sigma-clipping will be performed, and the
        time-series (with non-finite elems removed) will be passed through to
        the output.

    magsarefluxes : bool
        If your input measurements in `mags` are actually fluxes instead of
        mags, set this is True.

    verbose : bool
        If True, will indicate progress and report errors.

    Returns
    -------

    dict
        A dict of the following form is returned::

            {'itimes': the interpolated time values after gap-filling,
             'imags': the interpolated mag/flux values after gap-filling,
             'ierrs': the interpolated mag/flux values after gap-filling,
             'cadence': the cadence of the output mag/flux time-series,
             'minitime': the minimum value of the interpolated times array,
             'lags': the lags used to calculate the auto-correlation function,
             'acf': the value of the ACF at each lag used}

    '''

    # get the gap-filled timeseries
    interpolated = fill_magseries_gaps(times, mags, errs,
                                       fillgaps=fillgaps,
                                       forcetimebin=forcetimebin,
                                       sigclip=sigclip,
                                       magsarefluxes=magsarefluxes,
                                       filterwindow=filterwindow,
                                       verbose=verbose)

    if not interpolated:
        print('failed to interpolate light curve to minimum cadence!')
        return None

    itimes, imags = interpolated['itimes'], interpolated['imags'],

    # calculate the lags up to maxlags
    if maxlags:
        lags = nparange(0, maxlags)
    else:
        lags = nparange(itimes.size)

    series_stdev = 1.483*npmedian(npabs(imags))

    if func != _autocorr_func3:

        # get the autocorrelation as a function of the lag of the mag series
        autocorr = nparray([func(imags, x, imags.size, 0.0, series_stdev)
                            for x in lags])

    # this doesn't need a lags array
    else:

        autocorr = _autocorr_func3(imags, lags[0], imags.size,
                                   0.0, series_stdev)
        # return only the maximum number of lags
        if maxlags is not None:
            autocorr = autocorr[:maxlags]

    interpolated.update({'minitime':itimes.min(),
                         'lags':lags,
                         'acf':autocorr})

    return interpolated

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def acf_periodogram(time,
                    flux,
                    err,
                    maxlag_days=14.0,
                    cadence=2.0/1440.0,
                    height=0.05,
                    distance=10,
                    plot=True,
                    ax=None):

    import numpy as np
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    from scipy.signal import find_peaks
    from astrobase.varbase.autocorr import autocorr_magseries

    # ---------------------------------------------------------
    # Arrays contiguos
    # ---------------------------------------------------------
    x = np.ascontiguousarray(time, dtype=np.float64)
    y = np.ascontiguousarray(flux, dtype=np.float64)
    yerr = np.ascontiguousarray(err, dtype=np.float64)

    # Normalización tipo ppm (igual estilo que tu LS)
    mu = np.mean(y)
    y = (y / mu - 1) * 1e3
    yerr = yerr * 1e3 / mu

    # Limpiar no-finitos
    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr)
    x, y, yerr = x[mask], y[mask], yerr[mask]

    # ---------------------------------------------------------
    # ACF
    # ---------------------------------------------------------
    maxlags = int(maxlag_days / cadence)

    acf_out = autocorr_magseries(
        x,
        y,
        yerr,
        maxlags=maxlags,
        fillgaps=0.0,
        sigclip=None,
        magsarefluxes=True,
        verbose=False
    )

    if acf_out is None:
        raise RuntimeError("ACF failed")

    lags = acf_out["lags"]
    acf  = acf_out["acf"]
    cad  = acf_out["cadence"]

    lag_days = lags * cad

    # ---------------------------------------------------------
    # DataFrame tipo periodogram
    # ---------------------------------------------------------
    df_per = pd.DataFrame({
        "per": lag_days,
        "power": acf
    })

    # Buscar picos
    peaks, props = find_peaks(df_per["power"],
                              height=height,
                              distance=distance)

    df_peaks = (
        df_per.iloc[peaks]
        .sort_values(by="power", ascending=False)
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # Plot
    # ---------------------------------------------------------
    if plot:
        if ax is None:
            fig, ax = plt.subplots()
        else:
            fig = ax.figure

        ax.plot(df_per.per, df_per.power, c="black")
        ax.axhline(y=height, color="r", linestyle="--")
        sns.scatterplot(data=df_peaks,
                        x="per",
                        y="power",
                        ax=ax,
                        color="blue")

        ax.set_xlabel("Lag (days)")
        ax.set_ylabel("ACF")

        axins = ax.inset_axes([0.4, 0.4, 0.6, 0.6])
        axins.plot(df_per.per, df_per.power, c="black")
        axins.axhline(y=height, color="r", linestyle="--")
        sns.scatterplot(data=df_peaks,
                        x="per",
                        y="power",
                        ax=axins,
                        color="blue",
                        legend=False)

        axins.set_ylim(0, 0.3)
        axins.set_xlim(0, 5)
        axins.set_xticks([])
        axins.set_yticks([])
        axins.set_xlabel("")
        axins.set_ylabel("")
        return

    return df_peaks

In [5]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
def periodogram(time,flux,err,min_period=0.1,max_period=13,step=0.0001,tipo="LS",plot=True,ax=None):
    x = np.ascontiguousarray(time, dtype=np.float64)
    y = np.ascontiguousarray(flux, dtype=np.float64)
    yerr = np.ascontiguousarray(err, dtype=np.float64)
    mu = np.mean(y)
    y = (y / mu - 1) * 1e3
    yerr = yerr * 1e3 / mu
    min_freq = 1.0 / max_period #0.1
    max_freq = 1.0 / min_period
    freq = np.arange(min_freq, max_freq + step, step)
    if tipo=="LS":
        LS = LombScargle(x, y, yerr, normalization='standard')
        power = LS.power(freq)
        FP_99 = LS.false_alarm_level(0.0027,
                     minimum_frequency=freq[0],
                     maximum_frequency=freq[-1],
                     method='baluev')
        if FP_99 <= 0.01: level = 0.01
        if FP_99 > 0.01: level = FP_99+0.1*FP_99
    df_per = pd.DataFrame({"per":1./freq,"power":power})
    peaks, _ = find_peaks(df_per["power"], height=float(FP_99))
    df_peaks = df_per.iloc[peaks].sort_values(by="power",ascending=False).reset_index(drop=True)
    

    if plot:
        if ax is None:
            fig, ax = plt.subplots()
        else:
            fig = ax.figure

        ax.plot(df_per.per, df_per.power, c="black")
        ax.axhline(y=FP_99, color='r', linestyle='--')
        sns.scatterplot(data=df_peaks, x="per", y="power", ax=ax, color='blue')
        ax.set_xlabel("Period")
        ax.set_ylabel("Power")

        axins = ax.inset_axes([0.4, 0.4, 0.6, 0.6])
        axins.plot(df_per.per, df_per.power, c="black")
        axins.axhline(y=FP_99, color="r", linestyle="--")
        sns.scatterplot(data=df_peaks, x="per", y="power", ax=axins, color="blue", legend=False)
        axins.set_ylim(0, 0.2)
        axins.set_xlim(0, 5)
        axins.set_yticks([])
        axins.set_xticks([])
        axins.set_xlabel("")
        axins.set_ylabel("")
        return 
    return df_peaks

In [6]:
import numpy as np
from astropy.io import fits

def make_2d_histogram(n_bins_x, n_bins_y, data_mag, data_fase, norm_max="max"):
    bins_x = np.linspace(0, 1, n_bins_x)
    bins_y = np.linspace(data_mag.min(), data_mag.max(), n_bins_y)

    hist_data, _xbins, _ybins = np.histogram2d(
        data_fase, data_mag, bins=(bins_x, bins_y)
    )

    if norm_max == "log":
        # Normalización logarítmica análoga a LogNorm de imshow
        # log10(1 + x) evita problemas con ceros
        hist_data_log = np.log10(1 + hist_data)

        # Escalar a [0,1]
        max_val = hist_data_log.max()
        if max_val > 0:
            hist_data_norm = hist_data_log / max_val
        else:
            hist_data_norm = hist_data_log

        hist_data_transposed = hist_data_norm.T[::-1]
        return fits.PrimaryHDU(data=hist_data_transposed)

    if norm_max == "max":
        max_val = hist_data.max()
        hist_data_norm = hist_data / max_val if max_val > 0 else hist_data
        hist_data_transposed = hist_data_norm.T[::-1]
        return fits.PrimaryHDU(data=hist_data_transposed)

    if norm_max == "none":
        hist_data_transposed = hist_data.T
        return fits.PrimaryHDU(data=hist_data_transposed)

    else:
        norm_max = float(norm_max)
        hist_data = np.clip(hist_data, None, norm_max)
        hist_data_norm = hist_data / norm_max if norm_max > 0 else hist_data
        hist_data_transposed = hist_data_norm.T
        return fits.PrimaryHDU(data=hist_data_transposed)

In [7]:
import ray
ray.init()

@ray.remote
def save_lC(TIC,sector,path_save='/home/nicolas/nico/Data/TESS_LC_Massive_stars_NO_sigma_clipping'):
#open the LC always take long time bcs its necesary to apply the mask everytime    
    df = open_LC(TIC,sector, path_download='/home/nicolas/nico/git/OB-stars_LC/dowload_parallell/')
    df.to_csv(f"{path_save}/{TIC}_{sector}.csv",index=False)
    return 

    

2026-03-09 15:25:56,539	INFO worker.py:1752 -- Started a local Ray instance.


## Carga y filtrado del catálogo

In [8]:
df_1 = pd.read_csv("catalogs_and_data/3_MassiveXTessV8_LC_Aperture_corregido.csv")

In [9]:
df = pd.read_csv("/home/nicolas/nico/git/OB-stars_LC/catalogs_and_data/4_MassiveXTessV8_LC_Aperture_with_period_corregido_VSX.csv")

/tmp/ipykernel_12519/777874709.py:1: DtypeWarning: Columns (4,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/nicolas/nico/git/OB-stars_LC/catalogs_and_data/4_MassiveXTessV8_LC_Aperture_with_period_corregido_VSX.csv")


In [10]:
# Índices comunes entre df y df_1
common_indices = df_1["index"][df_1["index"].isin(df["index"])]

# Índices que están solo en df_1
only_in_df1 = df_1["index"][~df_1["index"].isin(df["index"])]


In [11]:
df_filtered = df[df["index"].isin(df_1["index"])]

In [12]:
sp_masivas = df[['sp_Li2021', 'sp_ALS', 'sp_GOSS', 'sp_IACOB']].notna().any(axis=1)

monsalves_masivas = (df["sp_Monsalves+2025"]=="B0-B3")|(df["sp_Monsalves+2025"]=="O-B0")

df.loc[(sp_masivas)|(df["label"]<=1.2)|(monsalves_masivas),"informacion_sp"] = 1

df = df.loc[(df["informacion_sp"]==1)|
        (df["B2_cut_probability"].isna())].reset_index(drop=True)

In [13]:
df = df.explode("sector_list").reset_index(drop=True)

## Descarga de curvas de luz y cómputo de periodogramas

In [14]:
cd CNN_periodogram/

/home/nicolas/nico/git/OB-stars_LC/CNN_periodogram


In [15]:
df = df.drop_duplicates(subset=["TIC","sector_list"]).reset_index(drop=True)

In [17]:
df.to_csv("CATALOG_VARIABILITY_TESS.csv",index=False)

In [13]:
futures = [
    save_lC.remote(df["TIC"][i], df["sector_list"][i]) for i in range(len(df))
]
results = ray.get(futures)

NameError: name 'save_lC' is not defined

In [ ]:
import ray
ray.shutdown()
ray.init()

@ray.remote
def check_number_peaks(TIC, sector, method="ACF"):

    import numpy as np

    # -----------------------------
    # Leer curva de luz
    # -----------------------------
    time, flux, flux_err = open_LC_csv(TIC, sector)

    per_min = np.diff(time).mean() * 2
    per_max = (time.max() - time.min()) / 2

    # -----------------------------
    # Selección de método
    # -----------------------------
    if method.upper() == "LS":

        df_peaks = periodogram(
            time,
            flux,
            flux_err,
            min_period=per_min,
            max_period=per_max,
            plot=False
        )

    elif method.upper() == "ACF":

        df_peaks = acf_periodogram(
            time,
            flux,
            flux_err,
            maxlag_days=per_max,
            cadence=np.diff(time).mean(),
            height=0.05,
            distance=10,
            plot=False
        )

    else:
        raise ValueError("method must be 'LS' or 'ACF'")

    # -----------------------------
    # Amplitud fotométrica
    # -----------------------------
    amplitude = np.abs(-2.5 * np.log10(flux.max() / flux.min()))

    # -----------------------------
    # Return solicitado
    # -----------------------------
    return df_peaks.per.values, list(df_peaks.power.values), amplitude

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
futures = [
    check_number_peaks.remote(df["TIC"][i], df["sector_list"][i],method="ACF") for i in range(len(df))
]

In [ ]:
results = ray.get(futures)

In [ ]:
period, power,amplitude = zip(*results)


df["period"] = period
df["power"] = power
df["amplitud_TESS"] = amplitude

In [ ]:
df = df.rename(columns={"Period":"per_vsx",
                   "Classification":"main_class_vsx","Type":"type_vsx"})

In [ ]:
df = df.explode(["period","power"]).reset_index(drop=True)

In [ ]:
df.to_csv("../catalogs_and_data/Github_MassiveXTessV8_LC_Aperture_Vsx_ACF.csv",index=False)

In [7]:
cd CNN_periodogram/

/home/nicolas/nico/git/OB-stars_LC/CNN_periodogram


In [8]:
df = pd.read_csv("../catalogs_and_data/Github_MassiveXTessV8_LC_Aperture_Vsx_LS.csv")

/tmp/ipykernel_75438/3061868708.py:1: DtypeWarning: Columns (4,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../catalogs_and_data/Github_MassiveXTessV8_LC_Aperture_Vsx_LS.csv")


In [9]:
ray.shutdown()

NameError: name 'ray' is not defined

In [10]:
import numpy as np
import pandas as pd

def open_LC_csv(
    TIC,
    sector,
    path_save='/home/nicolas/nico/Data/TESS_LC_Massive_stars/',
    undersampling=False,
    min_points=60,
    max_points=2000,
    seed=None
):
    df = pd.read_csv(f"{path_save}/{TIC}_{sector}.csv")
    
    time = df.Time.values
    flux = df.flux.values
    flux_err = df.flux_err.values

    if undersampling:
        if seed is not None:
            np.random.seed(seed)

        N = len(time)

        if N > min_points:
            n = np.random.randint(min_points, max_points + 1)
            n = min(n, N)

            idx = np.random.choice(N, size=n, replace=False)
            idx = np.sort(idx)  # mantiene orden temporal

            time = time[idx]
            flux = flux[idx]
            flux_err = flux_err[idx]

    return time, flux, flux_err

In [11]:
import ray
ray.init()


2026-03-02 14:47:39,913	INFO worker.py:1752 -- Started a local Ray instance.


Python version:,3.8.20
Ray version:,2.10.0


In [12]:
@ray.remote
def load_lc_data(tic_sector_pair):
    TIC, sector = tic_sector_pair
    return (TIC, sector, open_LC_csv(TIC, sector,undersampling=True))  # Retorna (TIC, sector, (time, flux, flux_error))

@ray.remote
def create_histogram(i, time, flux, per):
    fase = np.mod(time, per) / per
    hdu = make_2d_histogram(33, 33, flux, fase, norm_max="max")
    return (i, hdu.data)

In [13]:
# --- Preparar datos únicos ---
df["sector_list"] = df["sector_list"].astype(int)  # Asegurar tipo consistente
unique_pairs = list(set(zip(df["TIC"], df["sector_list"])))

# --- Cargar datos una sola vez por TIC/sector ---
futures_load = [load_lc_data.remote(pair) for pair in unique_pairs]
results_load = ray.get(futures_load)

# --- Construir diccionario de acceso rápido ---
lc_dict = {(TIC, sector): (time, flux, flux_error)
           for TIC, sector, (time, flux, flux_error) in results_load}

# --- Crear cubos paralelamente ---
futures = []
for i, (TIC, sector, per) in enumerate(zip(df["TIC"], df["sector_list"], df["period"])):
    if (TIC, sector) in lc_dict:
        time, flux, _ = lc_dict[(TIC, sector)]
        futures.append(create_histogram.remote(i, time, flux, per))

# --- Recoger resultados y construir el cubo final ---
results = ray.get(futures)
cubo = np.zeros((len(df), 32, 32, 1))
for i, data in results:
    cubo[i, :, :, 0] = data
    
np.save(f"../catalogs_and_data/Github_MassiveXTessV8_LC_Aperture_Vsx_LS_NO_sigma_cliping_min_max_undersampling.npy", cubo)

In [17]:
import tensorflow as tf

def make_model(output):
    model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(16, (3,3), input_shape=(32, 32, 1),activation="relu",padding="same"),
    tf.keras.layers.Conv2D(16, (3,3),activation="relu",padding="same"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(32, (3,3),activation="relu",padding="same"),
    tf.keras.layers.Conv2D(32, (3, 3),activation="relu",padding="same"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1024,activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(512,activation="relu"),
    tf.keras.layers.Dropout(0.3),
   # tf.keras.layers.Dense(1, activation='sigmoid')
    tf.keras.layers.Dense(output, activation='softmax')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=1e-4,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=0.1), loss="sparse_categorical_crossentropy", metrics=['acc'])
    return model

model_softmax = make_model(8)


2026-03-02 13:45:30.235216: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 13:45:30.280579: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-02 13:45:30.634806: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-02 13:45:30.637122: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 13:45:31.531897: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not fin

## Predict clases

In [1]:
cd CNN_periodogram/

/home/nicolas/nico/git/OB-stars_LC/CNN_periodogram


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import gdown
import joblib
import pandas as pd
import numpy as np
import tensorflow as tf
from tqdm import tqdm

In [3]:
def make_model():
    model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(16, (3,3), input_shape=(32, 32, 1),activation="relu",padding="same"),
    tf.keras.layers.Conv2D(16, (3,3),activation="relu",padding="same"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(32, (3,3),activation="relu",padding="same"),
    tf.keras.layers.Conv2D(32, (3, 3),activation="relu",padding="same"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1024,activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(512,activation="relu"),
    tf.keras.layers.Dropout(0.3),
   # tf.keras.layers.Dense(1, activation='sigmoid')
    tf.keras.layers.Dense(8, activation='softmax')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=1e-4,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=0.1), loss="sparse_categorical_crossentropy", metrics=['acc'])
    return model


def run_cnn_rf_pipeline(
    model_softmax,
    cubo,
    df,
    file_id_rf="1l_JSQ_VGArZ0oTKE1k48znzz87tGYeUR",
    rf_output="balanced_random_forest_model.joblib",
    weights_dir="/home/nicolas/nico/git/Paper_OGLE/Weights/batchBalanced_Number_M/",
    batch_size=256
):
    """
    Pipeline CNN + RF con taxonomía simplificada:
    Pulsating = M + CEP + DST + RR

    IMPORTANTE:
    - cubo y df deben estar en el MISMO orden.
    - No se eliminan filas.
    - No se rellenan NaNs.
    - Estrellas sin periodo => final_class = "No_periodo".
    """

    if len(cubo) != len(df):
        raise ValueError("cubo y df deben tener la misma longitud y orden.")

    # -----------------------------
    # Verificar weights
    # -----------------------------
    if not os.path.isdir(weights_dir):
        raise FileNotFoundError(
            f"\nNo se encontró el directorio:\n{weights_dir}\n"
            "Ejecuta:\n"
            "git clone https://github.com/Monsalves-Gonzalez-N/Paper_OGLE.git\n"
        )

    ckpt_path = tf.train.latest_checkpoint(weights_dir)
    if ckpt_path is None:
        raise FileNotFoundError(
            f"No se encontró ningún checkpoint (.ckpt) en {weights_dir}"
        )

    # -----------------------------
    # Descargar RF si no existe
    # -----------------------------
    if not os.path.exists(rf_output):
        gdown.download(
            f"https://drive.google.com/uc?id={file_id_rf}",
            rf_output,
            quiet=False
        )

    # -----------------------------
    # Cargar CNN
    # -----------------------------
    model_softmax.load_weights(ckpt_path)

    # -----------------------------
    # Predicción CNN (manteniendo orden)
    # -----------------------------
    preds = []
    n_samples = len(cubo)

    for i in tqdm(range(0, n_samples, batch_size), desc="CNN Predict"):
        batch = cubo[i:i + batch_size]
        batch_pred = model_softmax.predict(batch, verbose=0)
        preds.append(batch_pred)

    preds = np.vstack(preds)

    CNN_predictions = pd.DataFrame(
        preds,
        columns=['ELL', 'M', 'CEP', 'DST', 'E', 'LPV', 'RR', 'Rndm'],
        index=df.index
    )

    CNN_predictions["per"] = df["period"]
    CNN_predictions["amplitud"] = df["amplitud_TESS"]

    # -----------------------------
    # Random Forest
    # -----------------------------
    clf = joblib.load(rf_output)

    valid_mask = CNN_predictions["per"].notna()

    rf_columns = [
        'CNN+RF_ELL',
        'CNN+RF_M',
        'CNN+RF_CEP',
        'CNN+RF_DST',
        'CNN+RF_E',
        'CNN+RF_LPV',
        'CNN+RF_RR',
        'CNN+RF_Rndm'
    ]

    RF_and_CNN_predictions = pd.DataFrame(
        np.nan,
        index=df.index,
        columns=rf_columns
    )

    if valid_mask.any():
        X_valid = CNN_predictions.loc[valid_mask, clf.feature_names_in_]
        RF_probs = clf.predict_proba(X_valid)
        RF_and_CNN_predictions.loc[valid_mask, rf_columns] = RF_probs

    # -----------------------------
    # Simplificación taxonomía
    # Pulsating = M + CEP + DST + RR
    # -----------------------------
    final_class = pd.Series("No_periodo", index=df.index)

    if valid_mask.any():

        # Probabilidad agregada pulsating
        RF_and_CNN_predictions["CNN+RF_Pulsating"] = (
            RF_and_CNN_predictions[
                ['CNN+RF_M', 'CNN+RF_CEP', 'CNN+RF_DST', 'CNN+RF_RR']
            ].sum(axis=1)
        )

        reduced_cols = [
            'CNN+RF_ELL',
            'CNN+RF_E',
            'CNN+RF_LPV',
            'CNN+RF_Rndm',
            'CNN+RF_Pulsating'
        ]

        final_class.loc[valid_mask] = (
            RF_and_CNN_predictions.loc[valid_mask, reduced_cols]
            .idxmax(axis=1)
            .str.replace("CNN+RF_", "")
        )

    # -----------------------------
    # Guardar resultados finales
    # -----------------------------
    RF_and_CNN_predictions["final_class"] = final_class

    RF_and_CNN_predictions["CNN+RF_max_prob"] = np.nan
    if valid_mask.any():
        RF_and_CNN_predictions.loc[valid_mask, "CNN+RF_max_prob"] = (
            RF_and_CNN_predictions.loc[valid_mask, reduced_cols].max(axis=1)
        )

    return RF_and_CNN_predictions

In [4]:
df = pd.read_csv("../catalogs_and_data/Github_MassiveXTessV8_LC_Aperture_Vsx_LS.csv")
cubo = np.load("../catalogs_and_data/Github_MassiveXTessV8_LC_Aperture_Vsx_LS_NO_sigma_cliping_min_max_undersampling.npy")

/tmp/ipykernel_78440/2358456220.py:1: DtypeWarning: Columns (4,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../catalogs_and_data/Github_MassiveXTessV8_LC_Aperture_Vsx_LS.csv")


In [18]:
ls

106347848_summary_triple.pdf
114130098_39.png
11793277_41.png
125857387_66.png
139598582_62.png
141457686_summary_triple.pdf
251384128_58.png
260532127_57.png
308429909_63.png
314406170_65.png
328046141_summary_triple.pdf
336096991_summary_triple.pdf
337165095_71.png
339563367_66.png
34478136_summary_triple.pdf
346010525_58.png
362792232_summary_triple.pdf
391082033_summary_triple.pdf
400735422_summary_triple.pdf
418417840_57.png
430965138_57.png
440404514_64.png
443397649_64.png
458404466_63.png
458404466_64.png
458404466_summary_triple.pdf
459443701_64.png
58973594_66.png
balanced_random_forest_model.joblib
CATALOG_VARIABILITY_TESS.csv
CEP.pdf
ClassesCNN.pdf
CM_with_different_thresholds.pdf
CNN_RF_prediction_ACF_lineal.csv
CNN_RF_prediction_ACF_log.csv
CNN_RF_prediction_ACF_min_max.csv
CNN_RF_prediction_ACF_min_max_undersampling.csv
CNN_RF_prediction_LS_log.csv
CNN_RF_prediction_LS_min_max.csv
CNN_RF_prediction_LS_min_max_undersampling.csv
control_sample/
cube_pred_visually_check.npy

In [13]:
ACF_undersampling

,CNN+RF_ELL,CNN+RF_M,CNN+RF_CEP,CNN+RF_DST,CNN+RF_E,CNN+RF_LPV,CNN+RF_RR,CNN+RF_Rndm,CNN+RF_Pulsating,final_class,CNN+RF_max_prob,TIC,sector_list,power,period,amplitud_TESS
301079,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,Rndm,1.0,21671277,66,0.994170,0.013889,0.102079
114171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,Rndm,1.0,416544045,58,0.992100,0.032407,0.071281
103236,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,Rndm,1.0,141457686,61,0.991490,0.020833,0.209899
284369,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,Rndm,1.0,464631821,64,0.991388,0.030092,0.391871
231300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,Rndm,1.0,406252349,64,0.990654,0.041667,0.097517
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283880,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,No_periodo,NaN,193808452,56,NaN,NaN,0.001863
285548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,No_periodo,NaN,458295464,63,NaN,NaN,0.001680
289334,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,No_periodo,NaN,360548003,65,NaN,NaN,0.004561
291789,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,No_periodo,NaN,360841519,65,NaN,NaN,0.001763


In [20]:
rm /home/nicolas/Desktop/nico/


## predictions

In [ ]:
cd CNN_periodogram/

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../catalogs_and_data/3_MassiveXTessV8_LC_Aperture_Vsx_LS_CNN.csv")

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
TIC_irregulares = df.loc[df.groupby("TIC")["Rndm"].transform(lambda x: (x > 0.8).all())].sort_values(by="Rndm").drop_duplicates(subset=["TIC"])["TIC"]

In [ ]:
df = df.sort_values(by=['CNN+RF_max_prob',"power","Rndm"],
   ascending=[False,
  False,True]).drop_duplicates(subset=["TIC",
          "sector_list"])

In [ ]:
df.loc[df["CNN+RF_max_prob_class"]=="LPV"]

In [ ]:
df_non_per = pd.read_csv("../catalogs_and_data/3_MassiveXTessV8_LC_Aperture_Vsx_LS.csv")

In [ ]:
df_non_per = df_non_per.loc[df_non_per["period"].isna()].drop_duplicates(subset=["TIC",
          "sector_list"])

In [ ]:
df.loc[df["TIC"].isin(TIC_irregulares),"CNN+RF_max_prob_class"] = "Non-periodic"

In [ ]:
df_non_per = df_non_per.loc[df_non_per["period"].isna()].drop_duplicates(subset=["TIC",
          "sector_list"])

In [ ]:
df_non_per["CNN+RF_max_prob_class"] = "Non-periodic"

In [ ]:
df = pd.concat([df,df_non_per],axis=0).reset_index(drop=True)

In [ ]:
df.loc[df['CNN+RF_max_prob']<0.8,"CNN+RF_max_prob_class"] = "Uncertain"

In [ ]:
df[["CNN+RF_max_prob_class","TIC"]].groupby("CNN+RF_max_prob_class").count()

In [ ]:
df_final = df.sort_values(by=["CNN+RF_max_prob"],ascending=[False]).drop_duplicates(subset="TIC")

In [ ]:
df["CNN+RF_unique_count"] = df["TIC"].map(df.groupby("TIC")["CNN+RF_max_prob_class"].nunique())


In [ ]:
df.keys().values

In [ ]:
df.loc[(df["CNN+RF_max_prob_class"]=="E")&
        (df["label"]=>1.2)][["source_id","TIC","ra","dec","phot_g_mean_mag","skiff_type_list","label","CNN+RF_max_prob","Type","Period","period"]]

In [ ]:
df.loc[df["CNN+RF_unique_count"]==1].drop_duplicates(subset="TIC").groupby("CNN+RF_max_prob_class").count()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as path_effects
import numpy as np

# Datos
labels = ['Eclipsing', 'Ellipsoidal', 'Non-periodic', 'Pulsating', 'Unconstrain']
sizes = [1060, 1908, 47, 3375, 648]

# Paleta inferno_r
colors = plt.cm.inferno(np.linspace(0, 1, 5))

# Crear el gráfico de pastel
fig, ax = plt.subplots()
wedges, texts, autotexts = ax.pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'black'},
    textprops={'fontsize': 16}
)

# Estilo de texto
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_path_effects([
        path_effects.Stroke(linewidth=2, foreground='black'),
        path_effects.Normal()
    ])
    if autotext.get_text() == '2.9%':
        x, y = autotext.get_position()
        autotext.set_position((x + 0.25, y))

# Círculo perfecto
ax.axis('equal')

# Guardar el gráfico (ajusta path_plots si es necesario)
plt.savefig(f"Massive_stars_variability_distribution.pdf", format='pdf', bbox_inches='tight')

plt.show()



In [ ]:
df[["power","CNN+RF_max_prob_class","TIC"]].sort_values(by="power",ascending=False).drop_duplicates(subset=["TIC"]).groupby("CNN+RF_max_prob_class").count()

In [ ]:
df.loc[df["_RAJ2000"].isna()].drop_duplicates(subset="TIC")

In [ ]:
df.groupby(["TIC"])["CNN+RF_max_prob_class"].unique()

In [ ]:
471+925+1711+4+298

## Visualización y análisis de resultados

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
def periodogram(time,flux,err,min_period=0.1,max_period=13,step=0.0001,tipo="LS",plot=True,ax=None):
    x = np.ascontiguousarray(time, dtype=np.float64)
    y = np.ascontiguousarray(flux, dtype=np.float64)
    yerr = np.ascontiguousarray(err, dtype=np.float64)
    mu = np.mean(y)
    y = (y / mu - 1) * 1e3
    yerr = yerr * 1e3 / mu
    min_freq = 1.0 / max_period #0.1
    max_freq = 1.0 / min_period
    freq = np.arange(min_freq, max_freq + step, step)
    if tipo=="LS":
        LS = LombScargle(x, y, yerr, normalization='standard')
        power = LS.power(freq)
        FP_99 = LS.false_alarm_level(0.0027,
                     minimum_frequency=freq[0],
                     maximum_frequency=freq[-1],
                     method='baluev')
        if FP_99 <= 0.01: level = 0.01
        if FP_99 > 0.01: level = FP_99+0.1*FP_99
    df_per = pd.DataFrame({"per":1./freq,"power":power})
    peaks, _ = find_peaks(df_per["power"], height=float(FP_99))
    df_peaks = df_per.iloc[peaks].sort_values(by="power",ascending=False).reset_index(drop=True)
    

    if plot:
        if ax is None:
            fig, ax = plt.subplots()
        else:
            fig = ax.figure

        ax.plot(df_per.per, df_per.power, c="black")
        ax.axhline(y=FP_99, color='r', linestyle='--')
        sns.scatterplot(data=df_peaks, x="per", y="power", ax=ax, color='blue')
        ax.set_xlabel("Period")
        ax.set_ylabel("Power")

        axins = ax.inset_axes([0.4, 0.4, 0.6, 0.6])
        axins.plot(df_per.per, df_per.power, c="black")
        axins.axhline(y=FP_99, color="r", linestyle="--")
        sns.scatterplot(data=df_peaks, x="per", y="power", ax=axins, color="blue", legend=False)
        axins.set_ylim(0, 0.2)
        axins.set_xlim(0, 5)
        axins.set_yticks([])
        axins.set_xticks([])
        axins.set_xlabel("")
        axins.set_ylabel("")
        return 
    return df_peaks

In [ ]:
def plot_TIC_summary_triple(rows):
    fig, axes = plt.subplots(4, 3, figsize=(12, 15))

    for idx, row in enumerate(rows):
        time, flux, err = open_LC_csv(row["TIC"], row["sector_list"])
        period = row["Period"]
        clase = row["Predicted Class"]

        axes[idx, 0].errorbar(time - time.min(), flux - flux.mean(), yerr=err, fmt='.', markersize=2, c="black")
        axes[idx, 0].set_xlabel("Days", fontsize=15)

        per_min = np.diff(time).mean() * 2
        per_max = (time.max() - time.min()) / 2
        periodogram(time, flux, err, min_period=per_min, max_period=per_max,
                    tipo="LS", plot=True, ax=axes[idx, 1])
        axes[idx, 1].axvline(x=period, color='red', linestyle='--')
        axes[idx, 1].set_xlabel("P[d]", fontsize=15)
        
        phase = np.mod(time, period) / period
        axes[idx, 2].plot(phase, flux - flux.mean(), "k.", markersize=1)
        axes[idx, 2].set_xlabel("Phase", fontsize=15)

        # Remove default subplot titles
        axes[idx, 0].set_title("")
        axes[idx, 2].set_title("")
        row["Period"] = str(np.round(period,5)) +"d"
        if idx == len(rows) - 1:
            textstr = f"Predicted class {row['Predicted Class']}"
        else:
            info = row[['Predicted Class','Predicted Probability',"Period"]]
            textstr = '    '.join([f"{k}: {v}" for k, v in info.items()])
        axes[idx, 1].set_title(textstr, fontsize=18, pad=15,
                               bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
    fig.delaxes(axes[-1, -1])
    plt.tight_layout()
    filename = f"{rows[0]['TIC']}_summary_triple.pdf"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


In [ ]:
df = df.loc[df["CNN+RF_max_prob"]>=0.8].reset_index(drop=True)

In [ ]:
df = df.rename(columns={"CNN+RF_max_prob":'Predicted Probability',
                  "CNN+RF_max_prob_class":'Predicted Class',
                  "Period":"Period_vsx"})

In [ ]:
df = df.rename(columns={"period":"Period"})

In [ ]:
df_final = df[["TIC","Period","Predicted Class","Predicted Probability","sector_list","Type"]].reset_index(drop=True)

In [ ]:
len(df_final.loc[df_final["Type"].isna()].drop_duplicates(subset="TIC"))/len(df_final.drop_duplicates(subset="TIC"))

In [ ]:
df_final["num_pred_classes_per_sector"] = df_final.groupby("TIC")["Predicted Class"].transform(lambda x: x.nunique())
df_final["num_per_sector"] = df_final.groupby("TIC")["sector_list"].transform(len)

In [ ]:
df_final = df_final.loc[(df_final["num_per_sector"]>1)&
                (df_final["num_pred_classes_per_sector"]==1)]

In [ ]:
lista = []
for i in df_final["TIC"].unique():
    df_var = df_final.loc[df_final["TIC"]==i]
    lista.append(df_var["Period"].std()/df_var["Period"].mean())

In [ ]:
interesting = pd.DataFrame({"TIC":df_final["TIC"].unique(),"stdmean_per":lista})

In [ ]:
interesting.loc[interesting["stdmean_per"]>0.1]

In [ ]:
plt.hist(lista,bins=50,color="black")
plt.xlabel(r"$\sigma_P / \mu_P$ [d]",size=15)
plt.savefig("per_std.pdf", dpi=300)


In [ ]:
import numpy as np

grouped = df_final.groupby("TIC")["Period"]
ratio = grouped.apply(lambda x: np.std(x, ddof=0) / np.mean(x))
df_final = df_final.merge(ratio.rename("std_mean_ratio"), on="TIC", how="left")


In [ ]:
df_final.loc[df_final["TIC"]==72696935]

In [ ]:
df_final.loc[df_final["std_mean_ratio"]>0.2]

In [ ]:
df_final.loc[df_final["interesting"]>0.1]

In [ ]:
df_final.drop_duplicates(subset=["TIC"]).groupby("Predicted Class").count() / len(df_final.drop_duplicates(subset=["TIC"]))

In [ ]:
df  = df.rename(columns={"CNN+RF_max_prob_class":'Predicted Class',
                        "CNN+RF_max_prob":'Predicted Probability',
                        "Period":"period_vsx"})

In [ ]:
df  = df.rename(columns={"CNN+RF_max_prob_class":'Predicted Class',
                        "CNN+RF_max_prob":'Predicted Probability',
                        "period":"Period"})

In [ ]:

rows = [df.iloc[8], df.iloc[1559], df.iloc[295],df.iloc[7026]]

plot_TIC_summary_triple(rows)

In [ ]:
df_final = df.sort_values(by=["Predicted Probability","power"],ascending=[False,False]).drop_duplicates(subset="TIC")

In [ ]:
df_final[["TIC","Predicted Class"]].groupby("Predicted Class").count() 

In [ ]:
df.sort_values(by=["Predicted Probability","power"],ascending=[False,False])